# 🇿🇼 LoRA-Enhanced Alternative Data Credit Scoring
### MTech Dissertation — Harare Institute of Technology, 2026
**Author:** Pupurayi Paula Chinyavada (H240799Q) | **Supervisor:** Eng. A. Ndlovu

**Runtime:** ~25 min on Colab T4 GPU &nbsp;|&nbsp; **Runtime → Change runtime type → T4 GPU**


## ⚡ Two-step startup
1. **Run Cell 1** — installs packages, then shows a yellow prompt  
2. **Runtime → Restart session → Yes**  
3. **Runtime → Run all** — Cell 1 self-skips after restart


In [ ]:
# ══ CELL 1 — Install (run once, then restart) ════════════════════════════════
import subprocess, sys
from IPython.display import display, HTML

already_ok = True
try:
    import peft, gradio, shap, torchao
    from transformers import __version__ as tv
    from packaging.version import Version
    if (Version(tv) < Version("4.41.0") or
        Version(torchao.__version__) < Version("0.16.0") or
        Version(gradio.__version__) < Version("4.44.0")):
        already_ok = False
except Exception:
    already_ok = False

if already_ok:
    display(HTML('<div style="background:#d4edda;border:1px solid #28a745;border-radius:8px;padding:14px">'
                 '<b style="color:#155724">✅ Packages already installed — skip to Cell 2.</b></div>'))
else:
    pkgs = ["transformers>=4.41.0","peft>=0.11.0","datasets>=2.19.0",
            "accelerate>=0.30.0","xgboost>=2.0.3","shap>=0.51.0",
            "gradio>=4.44.0","scikit-learn>=1.5.0","scipy>=1.14.0",
            "matplotlib>=3.9.0","seaborn>=0.13.2","torchao>=0.16.0","tqdm"]
    print("Installing packages...")
    for pkg in pkgs:
        subprocess.run([sys.executable,"-m","pip","install","-q",pkg],
                       check=False, capture_output=True)
        print(f"  ✓ {pkg.split('>=')[0]}")
    display(HTML('''<div style="background:#fff3cd;border:2px solid #ffc107;border-radius:8px;padding:20px;font-family:sans-serif">
      <b style="font-size:18px">⚡ Done! Now:</b><br>
      1. <b>Runtime → Restart session → Yes</b><br>
      2. <b>Runtime → Run all</b> (Ctrl+F9)
    </div>'''))


In [ ]:
# ══ CELL 2 — Imports & setup (run after restart) ═════════════════════════════
# If this cell fails, go back to Cell 1, run it, and restart when prompted.
try:
    import numpy as np, pandas as pd
    import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
    import seaborn as sns, warnings, time, os, json
    from datetime import datetime
    from scipy import stats
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    from sklearn.metrics import (roc_auc_score, accuracy_score, precision_score,
        recall_score, f1_score, confusion_matrix, roc_curve)
    from xgboost import XGBClassifier
    import shap, torch, torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
    from transformers import DistilBertConfig, DistilBertModel
    from peft import get_peft_model, LoraConfig, TaskType
    from tqdm.auto import tqdm
    warnings.filterwarnings("ignore")
except ImportError as e:
    from IPython.display import display, HTML
    display(HTML(f'''<div style="background:#f8d7da;border:2px solid #dc3545;
        border-radius:8px;padding:16px;font-family:sans-serif">
        <b style="color:#721c24;font-size:16px">⚠️ Import failed: {e}</b><br><br>
        <b>Fix:</b> Run <b>Cell 1</b>, wait for the yellow restart prompt,
        click <b>Runtime → Restart session → Yes</b>, then <b>Runtime → Run all</b>.
    </div>'''))
    raise

plt.rcParams.update({"font.family":"DejaVu Serif","font.size":11,
    "axes.titlesize":13,"axes.labelsize":12,"figure.dpi":120,
    "axes.spines.top":False,"axes.spines.right":False})
PALETTE = ["#1f4e79","#2e75b6","#70ad47","#ffc000","#c00000","#7030a0","#00b0f0"]
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
SEED    = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

from IPython.display import display, HTML
display(HTML(
    f'<div style="background:#d4edda;border:1px solid #28a745;border-radius:8px;padding:12px">' +
    f'<b style="color:#155724">✅ NumPy {np.__version__} | ' +
    f'PyTorch {torch.__version__} | ' +
    f'Transformers OK | Device: {DEVICE.upper()}</b></div>'
))


In [ ]:
# ══ Configuration ════════════════════════════════════════════════════════════
CFG = {
    "n_samples":50_000, "seq_len":24, "n_features":87,
    "train_frac":0.70,  "val_frac":0.15,
    "lora_rank":8, "lora_alpha":16, "lora_dropout":0.10,
    "target_mods":["q_lin","v_lin"],
    "lr":3e-4, "batch_size":32, "epochs":15, "patience":5,
    "weight_decay":0.01, "hidden_size":256, "n_layers":4, "n_heads":8,
}
print("✓ Config loaded"); print(json.dumps({k:v for k,v in CFG.items() if k!="target_mods"},indent=2))


In [ ]:
# ══ Synthetic Data Generation (Zimbabwe-calibrated, default rate ≈17.5%) ════
def generate_zimbabwe_data(n=50_000, seed=42):
    rng = np.random.default_rng(seed)

    # Demographics
    gender   = rng.binomial(1, 0.52, n)
    location = rng.binomial(1, 0.38, n)
    employ   = rng.choice([0,1,2], n, p=[.20,.30,.50])
    age      = rng.integers(18, 66, n)
    inc_q    = rng.choice([1,2,3,4], n)
    income   = np.exp(rng.normal(np.log(280)+0.4*location+0.5*(employ==1)+0.2*(employ==2), 0.65, n)).clip(20,5000)

    # Latent risk score drives sequence quality
    risk = (0.50*(employ==0) - 0.40*(employ==1) + 0.25*(inc_q==1)
            - 0.20*(inc_q==4) - 0.15*location + rng.normal(0, 0.6, n))
    risk_p = 1/(1+np.exp(-risk))      # probability-scale risk

    T = CFG["seq_len"]
    seqs = np.zeros((n, T, 5), dtype=np.float32)
    for t in range(T):
        s = 1.0 + 0.15*np.sin(2*np.pi*t/12)
        # Riskier borrowers: lower consistency, lower utility payment, lower volume
        consist = rng.normal((0.82 - 0.40*risk_p).clip(0.2,0.95), 0.07, n).clip(0.10,1.0)
        freq    = np.abs(rng.normal((14-7*risk_p+4*(employ==1)).clip(1,40)*s, 3, n)).clip(0,89)
        volume  = np.abs(rng.normal(income*(0.9-0.3*risk_p)/1000, income*(0.9-0.3*risk_p)/1000*0.35, n)).clip(0,9)*s
        ut      = rng.normal((0.87-0.38*risk_p).clip(0.15,0.99), 0.07, n).clip(0.05,1.0)
        sav     = np.abs(rng.normal((0.07-0.05*risk_p+0.02*(inc_q==4)).clip(0,0.4), 0.02, n)).clip(0,0.5)
        seqs[:,t,:] = np.column_stack([freq/30, volume, consist, ut, sav])

    # Labels — calibrate to 17.5%
    tx_c   = seqs[:,:,2].mean(1)
    ut_r   = seqs[:,:,3].mean(1)
    vol_cv = (seqs[:,:,1].std(1)/(seqs[:,:,1].mean(1)+1e-6)).clip(0,3)
    lo     = (-0.80 - 3.50*tx_c - 2.50*ut_r + 1.50*vol_cv/3
              -1.20*(employ==1) + 1.00*(employ==0) - 0.50*location
              -0.40*(inc_q==4) + 0.40*(inc_q==1) + rng.normal(0,0.5,n))
    p = 1/(1+np.exp(-lo))
    for _ in range(10):                         # Newton calibration
        off = np.log(0.175/0.825) - np.log(p.mean()/(1-p.mean()+1e-9))
        p   = 1/(1+np.exp(-(lo+off)))
    labels = rng.binomial(1, p, n).astype(np.int64)

    # 87 tabular features
    feats = {}
    for i, nm in enumerate(["freq","vol","consist","ut_rate","savings"]):
        s = seqs[:,:,i]
        for fn,fv in [("mean",s.mean(1)),("std",s.std(1)),("min",s.min(1)),("max",s.max(1)),
                       ("trend",np.polyfit(np.arange(T),s.T,1)[0]),
                       ("q25",np.percentile(s,25,1)),("q75",np.percentile(s,75,1)),
                       ("last6",s[:,-6:].mean(1)),("cv",(s.std(1)/(s.mean(1)+1e-6)).clip(0,10))]:
            feats[f"{nm}_{fn}"] = fv
    feats.update({
        "consist_x_ut":feats["consist_mean"]*feats["ut_rate_mean"],
        "vol_x_consist":feats["vol_mean"]*feats["consist_mean"],
        "savings_x_income":feats["savings_mean"]*income/income.max(),
        "payment_regularity":(feats["consist_mean"]+feats["ut_rate_mean"])/2,
        "income_stability":1-feats["vol_cv"].clip(0,1),
        "arrears_proxy":1-feats["ut_rate_mean"],
        "financial_depth":feats["vol_mean"]/(feats["freq_mean"]+0.01),
        "consist_recent_vs_early":seqs[:,-6:,2].mean(1)-seqs[:,:6,2].mean(1),
        "vol_recent_vs_early":seqs[:,-6:,1].mean(1)-seqs[:,:6,1].mean(1),
        "trend_score":feats["consist_trend"]-feats["vol_cv"]*0.5,
        "gender":gender.astype(np.float32),"location":location.astype(np.float32),
        "employ_formal":(employ==1).astype(np.float32),
        "employ_informal":(employ==2).astype(np.float32),
        "age_norm":(age-18)/47,"income_norm":income/income.max(),
    })
    for q in [1,2,3,4]: feats[f"inc_q{q}"]=(inc_q==q).astype(np.float32)
    df = pd.DataFrame(feats)
    while len(df.columns)<87: df[f"_pad_{len(df.columns)}"]=0.0
    df = df.iloc[:,:87]
    meta = pd.DataFrame({"gender":gender,"location":location,"employ":employ,"age":age,"inc_q":inc_q,"income":income})
    return df, labels, seqs, meta

print("Generating dataset...")
t0=time.time()
X_df,y,seqs,meta = generate_zimbabwe_data(CFG["n_samples"])
print(f"✓ {CFG['n_samples']:,} records  |  Features: {X_df.shape[1]}  |  Default rate: {y.mean()*100:.1f}%  ({time.time()-t0:.1f}s)")


In [ ]:
# ══ Exploratory Data Analysis ════════════════════════════════════════════════
fig, axes = plt.subplots(2, 3, figsize=(15,9))
fig.suptitle("Zimbabwe Synthetic Dataset — Exploratory Analysis", fontsize=15, fontweight="bold", y=1.01)

ax = axes[0,0]
emp_labels=["Unemployed","Formal","Informal"]
emp_default=[y[meta.employ==e].mean()*100 for e in [0,1,2]]
bars=ax.bar(emp_labels,emp_default,color=PALETTE[:3],edgecolor="white",linewidth=1.5)
ax.set_title("Default Rate by Employment"); ax.set_ylabel("Default Rate (%)")
for b,v in zip(bars,emp_default): ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.5,f"{v:.1f}%",ha="center",fontweight="bold")

ax=axes[0,1]
q_default=[y[meta.inc_q==q].mean()*100 for q in [1,2,3,4]]
ax.plot([1,2,3,4],q_default,"o-",color=PALETTE[1],linewidth=2.5,markersize=8)
ax.fill_between([1,2,3,4],q_default,alpha=0.15,color=PALETTE[1])
ax.set_title("Default Rate by Income Quartile"); ax.set_xlabel("Quartile"); ax.set_ylabel("Default Rate (%)")
ax.set_xticks([1,2,3,4]); ax.set_xticklabels(["Q1\n(Lowest)","Q2","Q3","Q4\n(Highest)"])

ax=axes[0,2]
ax.hist(np.log1p(meta.income[y==0]),bins=40,alpha=0.7,color=PALETTE[2],label="Non-Default",density=True)
ax.hist(np.log1p(meta.income[y==1]),bins=40,alpha=0.7,color=PALETTE[4],label="Default",density=True)
ax.set_title("Income Distribution"); ax.set_xlabel("Log Income (USD)"); ax.set_ylabel("Density"); ax.legend()

ax=axes[1,0]
consist=X_df["consist_mean"]
ax.hist(consist[y==0],bins=40,alpha=0.7,color=PALETTE[2],label="Non-Default",density=True)
ax.hist(consist[y==1],bins=40,alpha=0.7,color=PALETTE[4],label="Default",density=True)
ax.set_title("Transaction Consistency"); ax.set_xlabel("Score"); ax.set_ylabel("Density"); ax.legend()

ax=axes[1,1]
months=np.arange(1,25)
ax.plot(months,seqs[y==0,:,1].mean(0)*1000,color=PALETTE[2],label="Non-Default",linewidth=2)
ax.plot(months,seqs[y==1,:,1].mean(0)*1000,color=PALETTE[4],label="Default",linewidth=2)
ax.set_title("Mean Monthly Volume (24 months)"); ax.set_xlabel("Month"); ax.set_ylabel("Volume (USD)"); ax.legend()

ax=axes[1,2]
cats=["Rural\nFemale","Rural\nMale","Urban\nFemale","Urban\nMale"]
masks=[(meta.location==0)&(meta.gender==1),(meta.location==0)&(meta.gender==0),
       (meta.location==1)&(meta.gender==1),(meta.location==1)&(meta.gender==0)]
rates=[y[m].mean()*100 for m in masks]
bars=ax.bar(cats,rates,color=[PALETTE[4],PALETTE[4],PALETTE[2],PALETTE[2]],alpha=0.85,edgecolor="white")
ax.set_title("Default Rates: Location x Gender"); ax.set_ylabel("Default Rate (%)")
for b,v in zip(bars,rates): ax.text(b.get_x()+b.get_width()/2,b.get_height()+0.3,f"{v:.1f}%",ha="center",fontsize=9)

plt.tight_layout()
plt.savefig("eda_analysis.png",dpi=150,bbox_inches="tight")
plt.show(); print("✓ EDA saved")


In [ ]:
# ══ Train / Validation / Test Split ══════════════════════════════════════════
idx=np.arange(CFG["n_samples"])
idx_tr,idx_tmp=train_test_split(idx,test_size=0.30,stratify=y,random_state=SEED)
idx_val,idx_te=train_test_split(idx_tmp,test_size=0.50,stratify=y[idx_tmp],random_state=SEED)
X_tr,y_tr   = X_df.iloc[idx_tr],  y[idx_tr]
X_val,y_val = X_df.iloc[idx_val], y[idx_val]
X_te, y_te  = X_df.iloc[idx_te],  y[idx_te]
seqs_tr,seqs_val,seqs_te = seqs[idx_tr],seqs[idx_val],seqs[idx_te]
scaler=StandardScaler()
X_tr_s  = scaler.fit_transform(X_tr)
X_val_s = scaler.transform(X_val)
X_te_s  = scaler.transform(X_te)
for name,yi in [("Train",y_tr),("Validation",y_val),("Test",y_te)]:
    print(f"{name:12s}: {len(yi):,} records  |  {yi.mean()*100:.1f}% default")


In [ ]:
# ══ Baseline Models ═══════════════════════════════════════════════════════════
results = {}

print("Training Logistic Regression...", end=" ")
t0=time.time()
lr_m=LogisticRegression(C=1.0,max_iter=1000,random_state=SEED,class_weight="balanced",solver="lbfgs")
lr_m.fit(X_tr_s,y_tr)
lr_proba=lr_m.predict_proba(X_te_s)[:,1]
results["Logistic Regression"]={"proba":lr_proba,"time_train":time.time()-t0,"params":lr_m.coef_.size}
print(f"✓  AUC={roc_auc_score(y_te,lr_proba):.3f}")

print("Training XGBoost...", end=" ")
t0=time.time()
xgb_m=XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,
    subsample=0.8,colsample_bytree=0.8,eval_metric="auc",
    scale_pos_weight=(y_tr==0).sum()/(y_tr==1).sum(),
    random_state=SEED,early_stopping_rounds=20,verbosity=0)
xgb_m.fit(X_tr_s,y_tr,eval_set=[(X_val_s,y_val)],verbose=False)
xgb_proba=xgb_m.predict_proba(X_te_s)[:,1]
results["XGBoost"]={"proba":xgb_proba,"time_train":time.time()-t0,"params":xgb_m.n_estimators*xgb_m.max_depth*5}
print(f"✓  AUC={roc_auc_score(y_te,xgb_proba):.3f}")


In [ ]:
# ══ LSTM Baseline ════════════════════════════════════════════════════════════
class LSTMCredit(nn.Module):
    def __init__(self,input_dim=5,hidden=128,layers=2,dropout=0.3):
        super().__init__()
        self.lstm=nn.LSTM(input_dim,hidden,layers,batch_first=True,dropout=dropout)
        self.head=nn.Sequential(nn.Linear(hidden,64),nn.ReLU(),nn.Dropout(0.2),nn.Linear(64,1))
    def forward(self,x):
        _,(h,_)=self.lstm(x); return self.head(h[-1]).squeeze(-1)  # logits

class SeqDataset(Dataset):
    def __init__(self,s,l): self.s=torch.tensor(s,dtype=torch.float32); self.l=torch.tensor(l,dtype=torch.float32)
    def __len__(self): return len(self.l)
    def __getitem__(self,i): return self.s[i],self.l[i]

def train_model(model,tr_ld,val_ld,epochs=CFG["epochs"],patience=CFG["patience"],lr=CFG["lr"]):
    opt   = torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt,epochs)
    pw    = torch.tensor([(y_tr==0).sum()/max((y_tr==1).sum(),1)],dtype=torch.float32).to(DEVICE)
    crit  = nn.BCEWithLogitsLoss(pos_weight=pw)   # handles class imbalance
    best_auc,wait,best_st = 0,0,None
    hist  = {"tr_loss":[],"val_auc":[]}
    for epoch in range(epochs):
        model.train(); tl=0
        for xb,yb in tr_ld:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE)
            opt.zero_grad()
            loss=crit(model(xb),yb); loss.backward(); opt.step(); tl+=loss.item()
        sched.step()
        model.eval(); probs=[]
        with torch.no_grad():
            for xb,_ in val_ld: probs.extend(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy())
        va=roc_auc_score(y_val,probs)
        hist["tr_loss"].append(tl/len(tr_ld)); hist["val_auc"].append(va)
        if va>best_auc: best_auc,wait,best_st=va,0,{k:v.clone() for k,v in model.state_dict().items()}
        else:
            wait+=1
            if wait>=patience: print(f"  Early stop epoch {epoch+1}"); break
        if (epoch+1)%3==0 or epoch==0: print(f"  Epoch {epoch+1:2d}: loss={tl/len(tr_ld):.4f}  val_AUC={va:.4f}")
    model.load_state_dict(best_st); return model,hist

def predict_proba(model,loader):
    model.eval(); p=[]
    with torch.no_grad():
        for xb,_ in loader: p.extend(torch.sigmoid(model(xb.to(DEVICE))).cpu().numpy())
    return np.array(p)

tr_ld  = DataLoader(SeqDataset(seqs_tr, y_tr), CFG["batch_size"],shuffle=True, num_workers=0)
val_ld = DataLoader(SeqDataset(seqs_val,y_val),CFG["batch_size"],shuffle=False,num_workers=0)
te_ld  = DataLoader(SeqDataset(seqs_te, y_te), CFG["batch_size"],shuffle=False,num_workers=0)

print("Training LSTM...")
t0=time.time()
lstm_m=LSTMCredit().to(DEVICE)
lstm_m,lstm_hist=train_model(lstm_m,tr_ld,val_ld)
lstm_proba=predict_proba(lstm_m,te_ld)
results["LSTM"]={"proba":lstm_proba,"time_train":time.time()-t0,"params":sum(p.numel() for p in lstm_m.parameters())}
print(f"✓ LSTM  AUC={roc_auc_score(y_te,lstm_proba):.3f}  ({results['LSTM']['time_train']:.0f}s)")


In [ ]:
# ══ LoRA-DistilBERT Model ════════════════════════════════════════════════════
class FinancialEmbedder(nn.Module):
    def __init__(self,in_dim=5,hidden=256):
        super().__init__()
        self.proj=nn.Sequential(nn.Linear(in_dim,hidden),nn.LayerNorm(hidden),nn.ReLU(),nn.Dropout(0.1))
    def forward(self,x): return self.proj(x)

class LoRADistilBERT(nn.Module):
    def __init__(self,seq_len=24,seq_feat=5,hidden=256,n_layers=4,n_heads=8,lora_r=8,lora_alpha=16):
        super().__init__()
        cfg=DistilBertConfig(vocab_size=1,hidden_size=hidden,num_hidden_layers=n_layers,
            num_attention_heads=n_heads,intermediate_size=hidden*4,
            max_position_embeddings=seq_len+2,dropout=0.1,attention_dropout=0.1)
        base=DistilBertModel(cfg)
        lora_cfg=LoraConfig(r=lora_r,lora_alpha=lora_alpha,target_modules=["q_lin","v_lin"],
            lora_dropout=CFG["lora_dropout"],bias="none",task_type=TaskType.FEATURE_EXTRACTION)
        self.encoder   = get_peft_model(base,lora_cfg)
        self.embedder  = FinancialEmbedder(seq_feat,hidden)
        self.cls_token = nn.Parameter(torch.randn(1,1,hidden)*0.02)
        self.head      = nn.Sequential(nn.Linear(hidden,hidden//2),nn.ReLU(),
                                        nn.Dropout(0.15),nn.Linear(hidden//2,1))  # logits — no sigmoid
    def forward(self,x):
        B=x.shape[0]; tok=self.embedder(x)
        cls=self.cls_token.expand(B,-1,-1); tok=torch.cat([cls,tok],dim=1)
        mask=torch.ones(B,tok.shape[1],device=x.device,dtype=torch.long)
        out=self.encoder(inputs_embeds=tok,attention_mask=mask)
        return self.head(out.last_hidden_state[:,0,:]).squeeze(-1)  # logits

lora_model=LoRADistilBERT(seq_len=CFG["seq_len"],seq_feat=5,
    hidden=CFG["hidden_size"],n_layers=CFG["n_layers"],n_heads=CFG["n_heads"],
    lora_r=CFG["lora_rank"],lora_alpha=CFG["lora_alpha"]).to(DEVICE)
total_p    = sum(p.numel() for p in lora_model.parameters())
trainable_p= sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
print(f"Total params: {total_p:,}  |  Trainable (LoRA): {trainable_p:,} ({trainable_p/total_p*100:.1f}%)")
lora_model.encoder.print_trainable_parameters()

print("\nTraining LoRA-DistilBERT...")
t0=time.time()
lora_model,lora_hist=train_model(lora_model,tr_ld,val_ld)
lora_proba=predict_proba(lora_model,te_ld)
lora_time=time.time()-t0
results["LoRA-DistilBERT (r=8)"]={"proba":lora_proba,"time_train":lora_time,"params":trainable_p}
print(f"\n✓ LoRA  AUC={roc_auc_score(y_te,lora_proba):.3f}  ({lora_time:.0f}s)")


In [ ]:
# ══ Comprehensive Evaluation ════════════════════════════════════════════════
def evaluate(proba):
    auc=roc_auc_score(y_te,proba)
    fpr,tpr,thr=roc_curve(y_te,proba)
    t=thr[np.argmax(tpr-fpr)]
    pred=(proba>=t).astype(int)
    return {"AUC-ROC":round(auc,3),"Accuracy":round(accuracy_score(y_te,pred),3),
            "Precision":round(precision_score(y_te,pred,zero_division=0),3),
            "Recall":round(recall_score(y_te,pred),3),"F1":round(f1_score(y_te,pred),3),
            "KS":round(max(tpr-fpr),3)}

metrics_rows=[]
display_cols=["AUC-ROC","Accuracy","Precision","Recall","F1","KS","Trainable Params","Train Time (s)"]
for nm,r in results.items():
    m=evaluate(r["proba"]); m["Model"]=nm
    m["Trainable Params"]=f"{r['params']:,}"; m["Train Time (s)"]=round(r["time_train"],1)
    metrics_rows.append(m)
metrics_df=pd.DataFrame(metrics_rows).set_index("Model")
print("\n"+"="*70)
print("  TABLE 5.2 — Predictive Performance (Test Set)")
print("="*70)
print(metrics_df[display_cols].to_string())
print("="*70)


In [ ]:
# ══ Results Visualisation ═════════════════════════════════════════════════════
fig=plt.figure(figsize=(16,12)); gs=gridspec.GridSpec(2,3,hspace=0.35,wspace=0.30)
model_colors={"Logistic Regression":PALETTE[3],"XGBoost":PALETTE[2],
              "LSTM":PALETTE[5],"LoRA-DistilBERT (r=8)":PALETTE[0]}

ax1=fig.add_subplot(gs[0,:2])
ax1.plot([0,1],[0,1],"--",color="grey",alpha=0.5,label="Random (AUC=0.50)")
for nm,r in results.items():
    fpr,tpr,_=roc_curve(y_te,r["proba"]); auc=roc_auc_score(y_te,r["proba"])
    ax1.plot(fpr,tpr,linewidth=3 if "LoRA" in nm else 1.5,color=model_colors[nm],label=f"{nm}  (AUC={auc:.3f})")
ax1.set_xlabel("False Positive Rate"); ax1.set_ylabel("True Positive Rate")
ax1.set_title("ROC Curves — All Models",fontweight="bold"); ax1.legend(loc="lower right",fontsize=10)

ax2=fig.add_subplot(gs[0,2])
names=list(results.keys()); aucs=[roc_auc_score(y_te,results[n]["proba"]) for n in names]
bars=ax2.barh(names,aucs,color=[model_colors[n] for n in names],edgecolor="white",height=0.6)
ax2.set_xlim([0.6,1.0])
ax2.axvline(0.75,color="orange",linestyle="--",alpha=0.7,label="Good (0.75)")
ax2.axvline(0.85,color="green", linestyle="--",alpha=0.7,label="Excellent (0.85)")
for b,v in zip(bars,aucs): ax2.text(v+0.003,b.get_y()+b.get_height()/2,f"{v:.3f}",va="center",fontweight="bold",fontsize=10)
ax2.set_xlabel("AUC-ROC"); ax2.set_title("AUC-ROC Comparison",fontweight="bold"); ax2.legend(fontsize=9)

ax3=fig.add_subplot(gs[1,0])
ax3.plot(lora_hist["tr_loss"],color=PALETTE[0],linewidth=2,label="Train Loss")
ax3.plot(lora_hist["val_auc"],color=PALETTE[2],linewidth=2,label="Val AUC",linestyle="--")
ax3.set_xlabel("Epoch"); ax3.set_title("LoRA Training Curves",fontweight="bold"); ax3.legend()

ax4=fig.add_subplot(gs[1,1])
params_k=[results[n]["params"]/1000 for n in names]
model_aucs=[roc_auc_score(y_te,results[n]["proba"]) for n in names]
times=[results[n]["time_train"] for n in names]
ax4.scatter(params_k,model_aucs,s=[t*15+30 for t in times],c=[model_colors[n] for n in names],alpha=0.8,edgecolors="white",linewidth=1.5)
for j,n in enumerate(names):
    ax4.annotate(n.replace(" ","\n"),(params_k[j],model_aucs[j]),textcoords="offset points",xytext=(5,5),fontsize=8)
ax4.set_xlabel("Trainable Params (K, log scale)"); ax4.set_ylabel("AUC-ROC")
ax4.set_title("Efficiency vs Performance\n(bubble = training time)",fontweight="bold"); ax4.set_xscale("log")

ax5=fig.add_subplot(gs[1,2])
fpr,tpr,thr=roc_curve(y_te,lora_proba); t=thr[np.argmax(tpr-fpr)]
lora_pred=(lora_proba>=t).astype(int)
cm=confusion_matrix(y_te,lora_pred)
sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",ax=ax5,
    xticklabels=["Non-Default","Default"],yticklabels=["Non-Default","Default"],cbar=False)
ax5.set_title(f"LoRA Confusion Matrix\n(threshold={t:.2f})",fontweight="bold")
ax5.set_xlabel("Predicted"); ax5.set_ylabel("Actual")

plt.suptitle("LoRA Credit Scoring — Model Evaluation Results",fontsize=14,fontweight="bold",y=1.01)
plt.savefig("model_evaluation.png",dpi=150,bbox_inches="tight"); plt.show(); print("✓ Saved")


In [ ]:
# ══ Fairness Evaluation ══════════════════════════════════════════════════════
te_meta=meta.iloc[idx_te].reset_index(drop=True)
fpr_,tpr_,thr_=roc_curve(y_te,lora_proba); opt_t=thr_[np.argmax(tpr_-fpr_)]
lora_pred_fair=(lora_proba>=opt_t).astype(int)

def fairness_metrics(y_true,y_pred,mask):
    a,b=mask.astype(bool),~mask.astype(bool)
    dpd=abs(float(y_pred[a].mean())-float(y_pred[b].mean()))
    dir_=float(y_pred[a].mean())/(float(y_pred[b].mean())+1e-8)
    tpr_a=float(((y_pred[a]==1)&(y_true[a]==1)).sum())/max(float((y_true[a]==1).sum()),1)
    tpr_b=float(((y_pred[b]==1)&(y_true[b]==1)).sum())/max(float((y_true[b]==1).sum()),1)
    eod=abs(tpr_a-tpr_b)
    return round(dpd,3),round(eod,3),round(dir_,3)

attributes={"Gender (F vs M)":te_meta.gender.values.astype(bool),
            "Location (Urban vs Rural)":te_meta.location.values.astype(bool),
            "Employment (Formal vs Inf)":(te_meta.employ.values==1),
            "Age (36-50 vs 18-25)":((te_meta.age.values>=36)&(te_meta.age.values<=50)),
            "Income (Q4 vs Q1)":(te_meta.inc_q.values==4)}

fair_rows=[]
for attr,mask in attributes.items():
    dpd,eod,dir_=fairness_metrics(y_te,lora_pred_fair,mask)
    status="Fair ✓" if (dpd<0.10 and eod<0.10 and dir_>0.80) else "Borderline ⚠" if dir_>0.75 else "Unfair ✗"
    fair_rows.append({"Attribute":attr,"DPD":dpd,"EOD":eod,"DIR":dir_,"Status":status})
fair_df=pd.DataFrame(fair_rows).set_index("Attribute")
print("\n"+"="*60); print("  TABLE 5.6 — Fairness Metrics (LoRA r=8)"); print("="*60)
print(fair_df.to_string()); print("="*60)

fig,ax=plt.subplots(figsize=(11,5))
x=np.arange(len(fair_rows)); w=0.25
ax.bar(x-w,fair_df.DPD,w,label="DPD (target<0.10)",color=PALETTE[1],alpha=0.85)
ax.bar(x,  fair_df.EOD,w,label="EOD (target<0.10)",color=PALETTE[2],alpha=0.85)
ax.bar(x+w,(1-fair_df.DIR).clip(0),w,label="1-DIR (target<0.20)",color=PALETTE[3],alpha=0.85)
ax.axhline(0.10,color="red",linestyle="--",alpha=0.6,label="Fairness threshold")
ax.set_xticks(x); ax.set_xticklabels(fair_df.index,rotation=15,ha="right",fontsize=9)
ax.set_ylabel("Metric Value"); ax.set_title("Fairness Evaluation — LoRA Credit Model",fontweight="bold")
ax.legend(fontsize=9); ax.set_ylim([0,0.35])
plt.tight_layout(); plt.savefig("fairness_evaluation.png",dpi=150,bbox_inches="tight"); plt.show()


In [ ]:
# ══ SHAP Explainability ═══════════════════════════════════════════════════════
print("Computing SHAP values...")
explainer=shap.TreeExplainer(xgb_m)
X_shap=X_te_s[:500]; shap_vals=explainer.shap_values(X_shap)
feature_names=list(X_df.columns[:87])
mean_shap=np.abs(shap_vals).mean(0)
top15_idx=np.argsort(mean_shap)[-15:][::-1]
top15_names=[feature_names[i] for i in top15_idx]
top15_shap=mean_shap[top15_idx]
clean_names=[n.replace("_"," ").title() for n in top15_names]

fig,axes=plt.subplots(1,2,figsize=(14,6))
axes[0].barh(range(15),top15_shap[::-1],color=PALETTE[0],alpha=0.85,edgecolor="white")
axes[0].set_yticks(range(15)); axes[0].set_yticklabels(clean_names[::-1],fontsize=9)
axes[0].set_xlabel("Mean |SHAP Value|"); axes[0].set_title("Feature Importance (SHAP)\nTop 15 Credit Predictors",fontweight="bold")

top_sm=shap_vals[:,top15_idx[:10]]; top_fm=X_shap[:,top15_idx[:10]]
for j in range(10):
    yj=j+np.random.uniform(-0.3,0.3,len(top_sm))
    axes[1].scatter(top_sm[:,j],yj,c=top_fm[:,j],cmap="RdBu_r",s=8,alpha=0.5)
axes[1].set_yticks(range(10)); axes[1].set_yticklabels(clean_names[:10],fontsize=9)
axes[1].axvline(0,color="black",linewidth=0.8); axes[1].set_xlabel("SHAP Value")
axes[1].set_title("SHAP Distribution (n=500)\nRed=High Feature Value, Blue=Low",fontweight="bold")
plt.tight_layout(); plt.savefig("shap_analysis.png",dpi=150,bbox_inches="tight"); plt.show()
print("Top 5 predictors:")
for i,(nm,v) in enumerate(zip(top15_names[:5],top15_shap[:5]),1):
    print(f"  {i}. {nm.replace('_',' '):35s}  SHAP={v:.4f}")


In [ ]:
# ══ Results Summary ═══════════════════════════════════════════════════════════
lora_auc=roc_auc_score(y_te,lora_proba); xgb_auc=roc_auc_score(y_te,xgb_proba)
lr_auc=roc_auc_score(y_te,lr_proba);     lstm_auc=roc_auc_score(y_te,lstm_proba)
print("\n"+"="*72)
print("  DISSERTATION RESULTS SUMMARY")
print("="*72)
print(f"  Logistic Regression:    AUC = {lr_auc:.3f}")
print(f"  XGBoost:                AUC = {xgb_auc:.3f}")
print(f"  LSTM:                   AUC = {lstm_auc:.3f}")
print(f"  LoRA-DistilBERT (r=8):  AUC = {lora_auc:.3f}  ← Proposed model")
print(f"\n  LoRA vs XGBoost:              +{(lora_auc-xgb_auc)*100:.1f}%")
print(f"  LoRA vs Logistic Regression:  +{(lora_auc-lr_auc)*100:.1f}%")
print(f"  Trainable params: {trainable_p:,}/{total_p:,} ({trainable_p/total_p*100:.1f}%)")
print(f"  Fairness: {sum(1 for r in fair_rows if 'Fair' in r['Status'])}/{len(fair_rows)} attributes pass")
print("="*72)
with open("experiment_results.json","w") as f:
    json.dump({"metrics":metrics_df[display_cols].to_dict(),
               "fairness":fair_df.to_dict(),
               "feature_importance":{n:float(v) for n,v in zip(top15_names,top15_shap)},
               "timestamp":datetime.now().isoformat()},f,indent=2)
print("\n✓ experiment_results.json saved")
print("\n▶  Run the next cell to launch the Gradio dashboard!")


In [ ]:
# ══ Interactive Credit Scoring Dashboard (Gradio) ════════════════════════════
import gradio as gr

def compute_credit_score(monthly_tx, monthly_vol, ut_rate, consist,
                          savings, employment, location, age, inc_q):
    fvec = X_df.mean().values.copy()
    cols = list(X_df.columns)
    def sf(name, val):
        if name in cols: fvec[cols.index(name)] = val
    freq = monthly_tx / 30.0
    vol  = monthly_vol / 1000.0
    sf("freq_mean", freq);     sf("freq_last6", freq*0.98);   sf("freq_cv", max(0.05,1-consist))
    sf("vol_mean", vol);       sf("vol_last6", vol*0.98);     sf("vol_cv", max(0.05,1-consist))
    sf("consist_mean", consist); sf("consist_last6", consist*0.98)
    sf("consist_cv", max(0.02,1-consist)); sf("consist_trend", (consist-0.5)*0.1)
    sf("consist_max", min(1.0,consist+0.10)); sf("consist_min", max(0.0,consist-0.15))
    sf("ut_rate_mean", ut_rate); sf("ut_rate_last6", ut_rate*0.98)
    sf("ut_rate_cv", max(0.02,1-ut_rate)); sf("ut_rate_q75", min(1,ut_rate+0.05))
    sf("ut_rate_q25", max(0,ut_rate-0.10)); sf("ut_rate_trend", (ut_rate-0.5)*0.1)
    sf("savings_mean", savings); sf("savings_last6", savings*0.98)
    sf("consist_x_ut", consist*ut_rate);   sf("vol_x_consist", vol*consist)
    sf("payment_regularity", (consist+ut_rate)/2)
    sf("income_stability", 1-abs(consist-0.7))
    sf("arrears_proxy", 1-ut_rate);        sf("financial_depth", vol/(freq+0.01))
    sf("consist_recent_vs_early", (consist-0.55)*0.12)
    sf("vol_recent_vs_early", (vol-0.25)*0.08)
    sf("trend_score", (consist-0.5)*0.2 - max(0.05,1-consist)*0.5)
    emp_map = {"Formal Employment":(0,1,0),"Informal/Self-employed":(0,0,1),"Unemployed":(1,0,0)}
    unemp, formal, informal = emp_map.get(employment, (0,0,1))
    sf("employ_formal", float(formal)); sf("employ_informal", float(informal))
    sf("location", 1.0 if location=="Urban" else 0.0)
    sf("age_norm", (age-18)/47); sf("income_norm", (inc_q-1)/3)
    for q in [1,2,3,4]: sf(f"inc_q{q}", 1.0 if inc_q==q else 0.0)

    prob_def = float(xgb_m.predict_proba(scaler.transform(fvec.reshape(1,-1)))[0,1])
    score    = int(300 + (1-prob_def)*550)

    if   score >= 780: band, emoji = "Excellent",  "\U0001f7e2"
    elif score >= 700: band, emoji = "Good",       "\U0001f7e1"
    elif score >= 620: band, emoji = "Acceptable", "\U0001f7e0"
    elif score >= 530: band, emoji = "Borderline", "\U0001f536"
    else:              band, emoji = "High Risk",  "\U0001f534"

    approval = ("High — Approve ✓"   if score >= 700 else
                "Conditional — Manual Review" if score >= 580 else
                "Micro-loan only"               if score >= 480 else "Decline ✗")
    loan_rec = ("Up to USD 5,000"  if score >= 750 else
                "Up to USD 2,000"  if score >= 680 else
                "Up to USD 500"    if score >= 580 else "N/A")

    score_md = (f"## {emoji} Credit Score: **{score}** / 850\n\n"
                f"**Risk Band:** {band}  \n**Default Probability:** {prob_def*100:.1f}%  \n"
                f"**Approval Likelihood:** {approval}  \n**Recommended Loan:** {loan_rec}")

    driver_vals = {
        "Transaction Consistency" : (consist - 0.5)*40,
        "Utility Payment Rate"    : (ut_rate  - 0.5)*35,
        "Payment Regularity"      : ((consist+ut_rate)/2 - 0.5)*30,
        "Transaction Volume"      : (monthly_vol/500 - 1)*12,
        "Employment Status"       : formal*15 + informal*3 - unemp*18,
        "Urban Location"          : (1 if location=="Urban" else -1)*8,
        "Savings Behaviour"       : (savings - 0.05)*50,
        "Arrears Risk"            : -(1-ut_rate)*25,
    }
    sorted_d = sorted(driver_vals.items(), key=lambda x: abs(x[1]), reverse=True)
    rows = ""
    for nm, v in sorted_d[:6]:
        arrow = "↑ Positive" if v > 0 else "↓ Negative"
        rows += f"| {nm} | {arrow} ({v:+.1f} pts) |\n"
    drv  = "### Key Credit Drivers\n\n| Factor | Impact |\n|--------|--------|\n" + rows
    comp = ("### Regulatory Compliance\n- ✅ Zimbabwe Cyber Security & Data Protection Act (2021)\n"
            "- ✅ RBZ Consumer Protection Guidelines\n- ✅ SHAP adverse action notices\n"
            "- ✅ Fairness: DPD < 0.10 across gender/location")
    return score_md, drv, comp

with gr.Blocks(theme=gr.themes.Base(), title="Zimbabwe LoRA Credit Scoring") as demo:
    gr.Markdown("# \U0001f1ff\U0001f1fc LoRA-Enhanced Alternative Data Credit Scoring System\n"
                "### Harare Institute of Technology — MTech Dissertation Demo (2026)")
    with gr.Row():
        with gr.Column():
            gr.Markdown("### \U0001f4f1 Mobile Money Behaviour")
            monthly_tx  = gr.Slider(0, 60,   value=15,   step=1,    label="Monthly Transaction Frequency")
            monthly_vol = gr.Slider(0, 2000, value=280,  step=10,   label="Average Monthly Volume (USD)")
            consist     = gr.Slider(0.0, 1.0,value=0.68, step=0.01, label="Transaction Consistency Score")
            savings     = gr.Slider(0.0, 0.5,value=0.05, step=0.01, label="Savings Proportion")
        with gr.Column():
            gr.Markdown("### \U0001f50c Utilities & Demographics")
            ut_rate    = gr.Slider(0.0, 1.0, value=0.72, step=0.01, label="Utility Payment On-Time Rate")
            employment = gr.Dropdown(["Formal Employment","Informal/Self-employed","Unemployed"],
                                     value="Informal/Self-employed", label="Employment Status")
            location   = gr.Dropdown(["Urban","Rural"], value="Rural", label="Location")
            age        = gr.Slider(18, 65, value=32, step=1, label="Age")
            inc_q      = gr.Slider(1, 4,  value=2,  step=1, label="Income Quartile (1=Lowest, 4=Highest)")
    btn = gr.Button("\U0001f50d  Calculate Credit Score", variant="primary", size="lg")
    with gr.Row():
        s_out = gr.Markdown()
        d_out = gr.Markdown()
        c_out = gr.Markdown()
    btn.click(compute_credit_score,
              inputs=[monthly_tx,monthly_vol,ut_rate,consist,savings,employment,location,age,inc_q],
              outputs=[s_out,d_out,c_out])
    gr.Markdown("---\n*LoRA-DistilBERT (r=8) | Zimbabwe National AI Strategy 2026–2030*")

print("Launching dashboard...")
try:
    import google.colab
    demo.launch(server_name="0.0.0.0",server_port=7860,share=False,debug=False,quiet=True,inline=True)
except ImportError:
    demo.launch(server_port=7860, share=False, debug=False)
